In [1]:
import os

from pathlib import Path

import numpy as np
import pandas as pd

from xgboost import XGBRegressor
from scipy.stats import spearmanr

In [9]:
import fastf1
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]  # adjust if needed
CACHE_DIR = PROJECT_ROOT / "data_raw" / "f1_cache"
fastf1.Cache.enable_cache(str(CACHE_DIR))
print("Cache:", CACHE_DIR)

Cache: c:\Users\aadha\OneDrive - Singapore Management University\Y2\S2\New folder\DAP\DAP F1 work\Working directory\data_raw\f1_cache


In [12]:
import pandas as pd
import numpy as np
import fastf1

# IMPORTANT: make sure cache is enabled somewhere above, e.g.
# fastf1.Cache.enable_cache("data_raw/f1_cache")

YEARS = list(range(2018, 2026))   # includes 2025
SESSION = "Q"

def td_to_ms(x):
    if pd.isna(x):
        return np.nan
    return x.total_seconds() * 1000.0

rows = []
failures = []

for year in YEARS:
    try:
        sched = fastf1.get_event_schedule(year)
    except Exception as e:
        failures.append((year, "schedule", type(e).__name__, str(e)))
        continue

    for _, ev in sched.iterrows():
        rnd = ev.get("RoundNumber")
        if pd.isna(rnd):
            continue
        rnd = int(rnd)

        # Skip testing / invalid schedule entries (round 0 shows up for testing)
        if rnd <= 0:
            continue

        event_name = ev.get("EventName", None)
        country = ev.get("Country", None)
        location = ev.get("Location", None)

        try:
            ses = fastf1.get_session(year, rnd, SESSION)
            ses.load(laps=True, telemetry=False, weather=False, messages=False)

            laps = ses.laps
            if laps is None or laps.empty:
                failures.append((year, rnd, "EmptyLaps", event_name))
                continue

            use = laps.copy()

            # 1) Keep only laps with a recorded LapTime (drops in/out laps)
            if "LapTime" in use.columns:
                use = use[use["LapTime"].notna()]

            # 2) Robust Deleted filtering (handles bool, 0/1, strings, NaN)
            if "Deleted" in use.columns:
                del_num = pd.to_numeric(use["Deleted"], errors="coerce")
                # Keep laps where Deleted is 0/False OR missing
                use = use[(del_num.isna()) | (del_num == 0)]

            if use.empty:
                failures.append((year, rnd, "NoUsableLaps", event_name))
                continue

            # Choose driver key robustly (FastF1 often uses DriverNumber in laps)
            if "DriverNumber" in use.columns:
                driver_key = "DriverNumber"
            elif "Driver" in use.columns:
                driver_key = "Driver"
            else:
                failures.append((year, rnd, "NoDriverKey", f"{event_name} missing Driver/DriverNumber"))
                continue

            # Convert timedelta columns -> ms
            for c in ["LapTime", "Sector1Time", "Sector2Time", "Sector3Time"]:
                if c in use.columns:
                    use[c + "_ms"] = use[c].apply(td_to_ms)

            # Aggregate per driver
            for drv, g in use.groupby(driver_key):
                out = {
                    "year": year,
                    "round": rnd,
                    "event_name": event_name,
                    "country": country,
                    "location": location,
                    "driver": str(drv),

                    "laps_count": int(g.shape[0]),
                    "laptime_best_ms": float(np.nanmin(g.get("LapTime_ms", np.nan))),
                    "laptime_med_ms": float(np.nanmedian(g.get("LapTime_ms", np.nan))),
                    "s1_best_ms": float(np.nanmin(g.get("Sector1Time_ms", np.nan))),
                    "s2_best_ms": float(np.nanmin(g.get("Sector2Time_ms", np.nan))),
                    "s3_best_ms": float(np.nanmin(g.get("Sector3Time_ms", np.nan))),
                }

                # speed traps if present
                for sp in ["SpeedI1", "SpeedI2", "SpeedFL", "SpeedST"]:
                    if sp in g.columns:
                        sp_num = pd.to_numeric(g[sp], errors="coerce")
                        out[f"{sp}_max"] = float(np.nanmax(sp_num))
                        out[f"{sp}_med"] = float(np.nanmedian(sp_num))

                # tyre info if present
                if "TyreLife" in g.columns:
                    out["tyrelife_max"] = float(np.nanmax(pd.to_numeric(g["TyreLife"], errors="coerce")))
                if "FreshTyre" in g.columns:
                    out["fresh_tyre_rate"] = float(pd.to_numeric(g["FreshTyre"], errors="coerce").fillna(0).mean())

                rows.append(out)

            print(f"[OK] {year} R{rnd:02d} {event_name} laps={len(use)} drivers={use[driver_key].nunique()}")

        except Exception as e:
            failures.append((year, rnd, type(e).__name__, str(e)))
            print(f"[FAIL] {year} R{rnd:02d} {event_name} -> {type(e).__name__}: {e}")

df_feat = pd.DataFrame(rows)
print("Built features:", df_feat.shape)
print("Failures:", len(failures))

if df_feat.empty:
    print("df_feat is empty. No rows were appended.")
    print("First 10 failures:", failures[:10])
else:
    print("Years:", sorted(df_feat["year"].unique()))
    display(df_feat.head())

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core           INFO 	Finished loading data for 20 drivers: ['44', '7', '5', '33', '3', '20', '8', '27', '55', '77', '14', '2', '11', '18', '31', '28', '9', '16', '35', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Usi

[OK] 2018 R01 Australian Grand Prix laps=199 drivers=20


core           INFO 	Finished loading data for 20 drivers: ['5', '7', '77', '44', '3', '10', '20', '27', '31', '55', '28', '11', '14', '2', '33', '8', '9', '35', '16', '18']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


[OK] 2018 R02 Bahrain Grand Prix laps=158 drivers=20


core        WARNING 	Fixed incorrect tyre stint information for driver '35'
core        WARNING 	Fixed incorrect tyre stint information for driver '18'
core           INFO 	Finished loading data for 20 drivers: ['5', '7', '77', '44', '33', '3', '27', '11', '55', '8', '20', '31', '14', '2', '28', '35', '10', '18', '16', '9']
core           INFO 	Loading data for Azerbaijan Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


...

core           INFO 	Finished loading data for 20 drivers: ['81', '4', '1', '63', '12', '6', '55', '14', '10', '16', '27', '30', '87', '5', '23', '22', '31', '44', '18', '43']
core           INFO 	Loading data for Abu Dhabi Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...


[OK] 2025 R23 Qatar Grand Prix laps=259 drivers=20


core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '63', '16', '14', '5', '31', '6', '22', '87', '55', '30', '12', '18', '44', '23', '27', '10', '43']


[OK] 2025 R24 Abu Dhabi Grand Prix laps=191 drivers=20
Built features: (3432, 22)
Failures: 0
Years: [np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]


,year,round,event_name,country,location,driver,laps_count,laptime_best_ms,laptime_med_ms,s1_best_ms,...,SpeedI1_max,SpeedI1_med,SpeedI2_max,SpeedI2_med,SpeedFL_max,SpeedFL_med,SpeedST_max,SpeedST_med,tyrelife_max,fresh_tyre_rate
0,2018,1,Australian Grand Prix,Australia,Melbourne,10,5,85295.0,108298.0,28146.0,...,276.0,246.0,288.0,269.0,299.0,296.0,310.0,249.0,4.0,1.0
1,2018,1,Australian Grand Prix,Australia,Melbourne,11,10,84005.0,93092.0,27618.0,...,284.0,272.0,295.0,291.5,306.0,300.0,323.0,278.5,4.0,1.0
2,2018,1,Australian Grand Prix,Australia,Melbourne,14,10,83597.0,94086.5,27489.0,...,277.0,269.0,291.0,285.5,299.0,297.0,309.0,279.5,4.0,1.0
3,2018,1,Australian Grand Prix,Australia,Melbourne,16,6,84636.0,95530.0,28153.0,...,281.0,267.0,293.0,285.0,303.0,298.5,315.0,256.5,2.0,1.0
4,2018,1,Australian Grand Prix,Australia,Melbourne,18,9,84230.0,106397.0,27793.0,...,282.0,258.0,293.0,274.0,298.0,296.0,311.0,262.0,4.0,1.0


In [14]:
label_rows = []
fail2 = []

for year in YEARS:
    try:
        sched = fastf1.get_event_schedule(year)
    except Exception:
        continue

    for _, ev in sched.iterrows():
        rnd = ev.get("RoundNumber")
        if pd.isna(rnd):
            continue
        rnd = int(rnd)

        if rnd <= 0:
            continue

        event_name = ev.get("EventName", None)

        try:
            ses = fastf1.get_session(year, rnd, SESSION)
            ses.load(laps=False, telemetry=False, weather=False, messages=False)

            res = ses.results
            if res is None or res.empty:
                continue

            for _, r in res.iterrows():
                label_rows.append({
                    "year": year,
                    "round": rnd,
                    "event_name": event_name,
                    "driver": str(r.get("DriverNumber")),  # KEY FIX
                    "quali_position": r.get("Position"),
                })

        except Exception as e:
            fail2.append((year, rnd, type(e).__name__, str(e)))

df_lab = pd.DataFrame(label_rows)
print(df_lab.shape, "label rows")
df_lab.head()

core           INFO 	Loading data for Australian Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['44', '7', '5', '33', '3', '20', '8', '27', '55', '77', '14', '2', '11', '18', '31', '28', '9', '16', '35', '10']
core           INFO 	Loading data for Bahrain Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['5', '7', '77', '44', '3', '10', '20', '27', '31', '55', '28', '11', '14', '2', '33', '8', '9', '35', '16', '18']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.7.0]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
core           INFO 	Finished loading data for 20 drivers: ['5', '7', '77', '44

(3459, 5) label rows


,year,round,event_name,driver,quali_position
0,2018,1,Australian Grand Prix,44,1.0
1,2018,1,Australian Grand Prix,7,2.0
2,2018,1,Australian Grand Prix,5,3.0
3,2018,1,Australian Grand Prix,33,4.0
4,2018,1,Australian Grand Prix,3,5.0


In [15]:
df = df_feat.merge(
    df_lab,
    on=["year", "round", "event_name", "driver"],
    how="inner"
)

print(df.shape)
df.head()

(3432, 23)


,year,round,event_name,country,location,driver,laps_count,laptime_best_ms,laptime_med_ms,s1_best_ms,...,SpeedI1_med,SpeedI2_max,SpeedI2_med,SpeedFL_max,SpeedFL_med,SpeedST_max,SpeedST_med,tyrelife_max,fresh_tyre_rate,quali_position
0,2018,1,Australian Grand Prix,Australia,Melbourne,10,5,85295.0,108298.0,28146.0,...,246.0,288.0,269.0,299.0,296.0,310.0,249.0,4.0,1.0,20.0
1,2018,1,Australian Grand Prix,Australia,Melbourne,11,10,84005.0,93092.0,27618.0,...,272.0,295.0,291.5,306.0,300.0,323.0,278.5,4.0,1.0,13.0
2,2018,1,Australian Grand Prix,Australia,Melbourne,14,10,83597.0,94086.5,27489.0,...,269.0,291.0,285.5,299.0,297.0,309.0,279.5,4.0,1.0,11.0
3,2018,1,Australian Grand Prix,Australia,Melbourne,16,6,84636.0,95530.0,28153.0,...,267.0,293.0,285.0,303.0,298.5,315.0,256.5,2.0,1.0,18.0
4,2018,1,Australian Grand Prix,Australia,Melbourne,18,9,84230.0,106397.0,27793.0,...,258.0,293.0,274.0,298.0,296.0,311.0,262.0,4.0,1.0,14.0


In [16]:
df.isna().mean().sort_values(ascending=False).head(10)

SpeedFL_med       0.001748
SpeedFL_max       0.001748
quali_position    0.000291
country           0.000000
year              0.000000
round             0.000000
event_name        0.000000
laps_count        0.000000
driver            0.000000
location          0.000000
dtype: float64

In [17]:
df["quali_position"].describe()

count    3431.000000
mean       10.412416
std         5.717638
min         1.000000
25%         5.000000
50%        10.000000
75%        15.000000
max        20.000000
Name: quali_position, dtype: float64

In [18]:
# df is your merged dataframe (df_feat + df_lab)
TARGET = "quali_position"

# Keep a copy of identifiers for later analysis
ID_COLS = ["year", "round", "event_name", "country", "location", "driver"]

# Drop rows without target
df_model = df.dropna(subset=[TARGET]).copy()
df_model[TARGET] = df_model[TARGET].astype(int)

print("df_model:", df_model.shape)
df_model[ID_COLS + [TARGET]].head()

df_model: (3431, 23)


,year,round,event_name,country,location,driver,quali_position
0,2018,1,Australian Grand Prix,Australia,Melbourne,10,20
1,2018,1,Australian Grand Prix,Australia,Melbourne,11,13
2,2018,1,Australian Grand Prix,Australia,Melbourne,14,11
3,2018,1,Australian Grand Prix,Australia,Melbourne,16,18
4,2018,1,Australian Grand Prix,Australia,Melbourne,18,14


In [ ]:
x

train/eval/test: (2477, 23) (478, 23) (476, 23)


In [21]:
# Candidate numeric features (drop target + identifiers)
drop_cols = set(ID_COLS + [TARGET])
num_cols = [c for c in df_model.columns if c not in drop_cols and pd.api.types.is_numeric_dtype(df_model[c])]

X_train, y_train = train_df[num_cols], train_df[TARGET]
X_eval,  y_eval  = eval_df[num_cols],  eval_df[TARGET]
X_test,  y_test  = test_df[num_cols],  test_df[TARGET]

print("num feature count:", len(num_cols))
print("example features:", num_cols[:10])

num feature count: 16
example features: ['laps_count', 'laptime_best_ms', 'laptime_med_ms', 's1_best_ms', 's2_best_ms', 's3_best_ms', 'SpeedI1_max', 'SpeedI1_med', 'SpeedI2_max', 'SpeedI2_med']


In [23]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr
import numpy as np

xgb = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.9,
    colsample_bytree=0.9,
    reg_lambda=1.0,
    random_state=42,
    n_jobs=-1
)

xgb.fit(
    X_train, y_train,
    eval_set=[(X_eval, y_eval)],
    verbose=False
)

pred_eval = xgb.predict(X_eval)
pred_test = xgb.predict(X_test)

def report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = float(np.sqrt(mse))
    sp = spearmanr(y_true, y_pred).correlation
    print(f"{name} | MAE={mae:.3f} RMSE={rmse:.3f} Spearman={sp:.3f}")

report("EVAL", y_eval, pred_eval)
report("TEST", y_test, pred_test)

EVAL | MAE=3.021 RMSE=3.752 Spearman=0.741
TEST | MAE=3.412 RMSE=4.382 Spearman=0.672


In [25]:
import json
from pathlib import Path
import joblib
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

ART_DIR = Path("artifacts") / "phase3" / "xgb_vanilla_eval2024_test2025"
ART_DIR.mkdir(parents=True, exist_ok=True)

# predictions with identifiers
preds_test = test_df[ID_COLS + [TARGET]].copy()
preds_test["pred_position"] = pred_test
preds_test.to_csv(ART_DIR / "predictions_test.csv", index=False)

preds_eval = eval_df[ID_COLS + [TARGET]].copy()
preds_eval["pred_position"] = pred_eval
preds_eval.to_csv(ART_DIR / "predictions_eval.csv", index=False)

def calc_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    mse = float(mean_squared_error(y_true, y_pred))
    rmse = float(np.sqrt(mse))
    sp = float(spearmanr(y_true, y_pred).correlation)
    return {"mae": mae, "rmse": rmse, "spearman": sp}

metrics = {
    "train_years": TRAIN_YEARS,
    "eval_year": EVAL_YEAR,
    "test_year": TEST_YEAR,
    "n_features": int(len(num_cols)),
    "features": list(num_cols),
    "eval": calc_metrics(y_eval, pred_eval),
    "test": calc_metrics(y_test, pred_test),
    "model_params": xgb.get_params(),
}

with open(ART_DIR / "metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

joblib.dump(xgb, ART_DIR / "model.joblib")

print("Saved artifacts to:", ART_DIR)
print("Eval metrics:", metrics["eval"])
print("Test metrics:", metrics["test"])

Saved artifacts to: artifacts\phase3\xgb_vanilla_eval2024_test2025
Eval metrics: {'mae': 3.0208120346069336, 'rmse': 3.751827049222204, 'spearman': 0.7406596843444286}
Test metrics: {'mae': 3.412320613861084, 'rmse': 4.3820710940563545, 'spearman': 0.6718407550524931}
